In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import warnings 
warnings.filterwarnings('ignore')
import re
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import hstack

# Scikit-learn modules
from sklearn.model_selection import StratifiedKFold, cross_val_score, RandomizedSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import RobustScaler, LabelEncoder, MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Models
from sklearn.linear_model import LogisticRegression, SGDClassifier, PassiveAggressiveClassifier, PassiveAggressiveClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

In [ ]:
train = pd.read_csv('/kaggle/input/mlp-term-3-2025-kaggle-assignment-3/train.csv')
test = pd.read_csv("/kaggle/input/mlp-term-3-2025-kaggle-assignment-3/test.csv")
sample_submission = pd.read_csv('/kaggle/input/mlp-term-3-2025-kaggle-assignment-3/sample_submission.csv')
train.head()

# 1. Identify data types of different columns

In [ ]:
train.info()

| Datatypes       | No of Features |
|-----------------|---------------|
| Int64           | 02            |
| Object          | 01            |
| Float64    | 03           |


# 2. Present descriptive statistics of numerical columns

In [ ]:
train.drop('id',axis=1).describe()

In [ ]:
test.drop('id',axis=1).describe()

# 3. Identify and handle the missing values

In [ ]:
train.isnull().sum()

In [ ]:
test.isnull().sum()

In [ ]:
# 1. Handle Unknown Numerical Features (Fill with -1, add flag)
num_cols = ['feature_1', 'feature_2', 'feature_3']
for col in num_cols:
    train[f'{col}_nan'] = train[col].isna().astype(int)
    test[f'{col}_nan'] = test[col].isna().astype(int)
    train[col] = train[col].fillna(-1)
    test[col] = test[col].fillna(-1)
    # Log transform to handle high variance
    train[col] = np.log1p(train[col] - train[col].min() + 1)
    test[col] = np.log1p(test[col] - test[col].min() + 1)

# 2. Text Cleaning
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"n't", " not", text)
    text = re.sub(r"'re", " are", text)
    text = re.sub(r"'s", " is", text)
    return text

train['clean_phrase'] = train['phrase'].apply(clean_text)
test['clean_phrase'] = test['phrase'].apply(clean_text)

y = train['sentiment']

# 4. Identify and handle duplicates

In [ ]:
# Duplicates in training data 
print("\nChecking for duplicate rows in training data...")
duplicate_count = train.duplicated().sum()
print(f"Duplicate rows found: {duplicate_count}")

if duplicate_count > 0:
    train = train.drop_duplicates().reset_index(drop=True)
    print(f"Duplicates removed. New train shape: {train.shape}")
else:
    print("No duplicates found.")

# Duplicates in test data
print("\nChecking for duplicate rows in Test data...")
duplicate_count = train.duplicated().sum()
print(f"Duplicate rows found: {duplicate_count}")

if duplicate_count > 0:
    train = train.drop_duplicates().reset_index(drop=True)
    print(f"Duplicates removed. New test shape: {train.shape}")
else:
    print("No duplicates found.")

# 5. Identify and handle outliers

In [ ]:
num_cols = ['feature_1','feature_2','feature_3']

In [ ]:
def clip_outliers(df, columns):
    summary_data = []
    for col in columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR

        # Count outliers
        outliers = df[(df[col] < lower) | (df[col] > upper)][col]
        count = outliers.count()
        pct = round((count / len(df)) * 100, 2)

        summary_data.append({
        'Column': col,
        'Outlier Count': count,
        'Outlier %': f"{pct:.2f}%",
        'lower_bound': f"{lower:.2f}",
        'Upper_bound': f"{upper:.2f}"
        })
        summary_df = pd.DataFrame(summary_data)
        # Clip
        df[col] = df[col].clip(lower, upper)

    return df, summary_df

train, train_report = clip_outliers(train, num_cols)
test, test_report = clip_outliers(test, num_cols)

# To see the results:
test_report

In [ ]:
train_report

# 6. Present at least three visualizations and provide insights for the same

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


train_copy = train.copy()

# 01 
sns.set_theme(style="whitegrid")
plt.figure(figsize=(10, 6))
ax = sns.countplot(data=train_copy, x='sentiment', palette='viridis')
plt.title('Distribution of Sentiment Classes', fontsize=15)
plt.xlabel('Sentiment (0=Negative, 1=Neutral, 2=Positive)', fontsize=12)
plt.ylabel('Count', fontsize=12)

# Add count labels
for p in ax.patches:
    ax.annotate(
        f'{p.get_height()}',
        (p.get_x() + p.get_width() / 2., p.get_height()),
        ha='center', va='baseline', fontsize=11,
        color='black', xytext=(0, 5), textcoords='offset points'
    )
plt.show()


# 02 
# Create word_count ONLY in the copy
train_copy['word_count'] = train_copy['phrase'].apply(lambda x: len(str(x).split()))

plt.figure(figsize=(10, 6))
sns.kdeplot(
    data=train_copy,
    x='word_count',
    hue='sentiment',
    fill=True,
    palette='crest',
    common_norm=False
)
plt.title('Density of Word Counts by Sentiment', fontsize=15)
plt.xlim(0, 50)
plt.xlabel('Number of Words in Phrase', fontsize=12)
plt.ylabel('Density', fontsize=12)
plt.legend(title='Sentiment', labels=['Positive (2)', 'Neutral (1)', 'Negative (0)'])
plt.show()


# 03 
melted_df = train_copy.melt(
    id_vars=['sentiment'],
    value_vars=['feature_1', 'feature_2', 'feature_3'],
    var_name='feature_name',
    value_name='feature_value'
)

plt.figure(figsize=(12, 6))
sns.boxplot(
    data=melted_df,
    x='feature_name',
    y='feature_value',
    hue='sentiment',
    palette='rocket'
)
plt.title('Distribution of Unknown Features by Sentiment', fontsize=15)
plt.yscale('log')
plt.xlabel('Features', fontsize=12)
plt.ylabel('Feature Value (Log Scale)', fontsize=12)
plt.legend(title='Sentiment', loc='upper right')
plt.show()

# 7. Scale Numerical features and Encode Categorical features

In [ ]:
print("\n--- Scaling and Encoding ---")

# 1. Scale Numerical Data
# We use RobustScaler as it is better for data with outliers
scaler = MinMaxScaler() 
num_features_list = num_cols + [c+'_nan' for c in num_cols]

# Fit and transform
X_train_num = scaler.fit_transform(train[num_features_list])
X_test_num = scaler.transform(test[num_features_list])

print("Numerics Scaled (0 to 1 range).")

# 2. Encode Text Data (The "Categorical" part)
# Strategy: Dual TF-IDF (Word + Character N-Grams) for maximum performance

# A. Word Vectorizer (1-3 words)
word_vectorizer = TfidfVectorizer(
    sublinear_tf=True,
    strip_accents='unicode',
    analyzer='word',
    ngram_range=(1, 3), 
    max_features=15000
)

# B. Char Vectorizer (3-6 chars) - captures subwords
char_vectorizer = TfidfVectorizer(
    sublinear_tf=True,
    strip_accents='unicode',
    analyzer='char',
    ngram_range=(3, 6), 
    max_features=25000
)

# Fit on combined corpus to ensure consistent vocabulary
all_text = pd.concat([train['clean_phrase'], test['clean_phrase']])
word_vectorizer.fit(all_text)
char_vectorizer.fit(all_text)

# Transform
X_train_word = word_vectorizer.transform(train['clean_phrase'])
X_test_word = word_vectorizer.transform(test['clean_phrase'])

X_train_char = char_vectorizer.transform(train['clean_phrase'])
X_test_char = char_vectorizer.transform(test['clean_phrase'])

print("Text Vectorized (Words + Chars).")

# 3. Combine All Features
# Result is a sparse matrix with ~40k columns
X_train_final = hstack([X_train_word, X_train_char, X_train_num])
X_test_final = hstack([X_test_word, X_test_char, X_test_num])

print(f"Final Training Data Shape: {X_train_final.shape}")

# 8. Model Building (at least 7)

In [ ]:
print("\n--- Model Building (Training & Evaluation) ---")

models = {
    "Logistic Regression": LogisticRegression(C=2.0, max_iter=1000, solver='sag', n_jobs=-1),
    "Linear SVC": LinearSVC(C=0.5, dual=False, random_state=42),
    "Multinomial NB": MultinomialNB(),
    "SGD Classifier": SGDClassifier(loss='hinge', penalty='l2', alpha=1e-4, random_state=42, n_jobs=-1),
    # REPLACED RIDGE WITH PASSIVE AGGRESSIVE TO FIX CRASH
    "Passive Aggressive": PassiveAggressiveClassifier(max_iter=1000, C=1.0, random_state=42, n_jobs=-1), 
    "LightGBM": LGBMClassifier(n_estimators=100, learning_rate=0.05, random_state=42, verbose=-1),
    "XGBoost": XGBClassifier(n_estimators=100, learning_rate=0.05, eval_metric='mlogloss', n_jobs=-1)
}

model_results = []

# We use Cross-Validation to get a reliable score
kf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

for name, model in models.items():
    print(f"Training {name}...")
    scores = cross_val_score(model, X_train_final, y, cv=kf, scoring='accuracy', n_jobs=-1)
    mean_score = scores.mean()
    model_results.append({'Model': name, 'Accuracy': mean_score})
    print(f"   -> {name} Average Accuracy: {mean_score:.4f}")

# 9. Hyperparameter Tuning on any 3 of the models

In [ ]:
print("\n--- Hyperparameter Tuning (Top 3 Models) ---")

# Convert results to DataFrame to find top 3
results_df = pd.DataFrame(model_results)
top_3_names = results_df.sort_values(by='Accuracy', ascending=False).head(3)['Model'].tolist()
print(f"Top 3 Models selected for tuning: {top_3_names}")

best_estimators = {}

for name in top_3_names:
    print(f"\nTuning {name}...")
    
    # Define search space based on the model name
    if name == "Logistic Regression":
        estimator = LogisticRegression(max_iter=1000, solver='sag', n_jobs=-1)
        param_dist = {'C': [0.1, 1.0, 2.0, 5.0, 10.0]}
        
    elif name == "Linear SVC":
        estimator = LinearSVC(dual=False, random_state=42)
        param_dist = {'C': [0.1, 0.5, 1.0, 2.0, 5.0]}
        
    elif name == "LightGBM":
        estimator = LGBMClassifier(random_state=42, verbose=-1)
        param_dist = {
            'n_estimators': [100, 200, 300],
            'learning_rate': [0.01, 0.05, 0.1],
            'num_leaves': [20, 31, 50]
        }
    
    elif name == "XGBoost":
        estimator = XGBClassifier(eval_metric='mlogloss', n_jobs=-1)
        param_dist = {
            'n_estimators': [100, 200, 300],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth': [3, 5, 7]
        }
        
    elif name == "SGD Classifier":
        estimator = SGDClassifier(max_iter=1000, n_jobs=-1)
        param_dist = {'alpha': [1e-4, 1e-3, 1e-2], 'loss': ['hinge', 'log_loss']}

    elif name == "Passive Aggressive":
        estimator = PassiveAggressiveClassifier(max_iter=1000, n_jobs=-1)
        # C controls regularization (similarity to SVM/Ridge)
        param_dist = {'C': [0.01, 0.1, 0.5, 1.0, 5.0], 'loss': ['hinge', 'squared_hinge']}

    else: # Fallback for MultinomialNB
        estimator = MultinomialNB()
        param_dist = {'alpha': [0.1, 0.5, 1.0]}

    # Run Randomized Search
    clf = RandomizedSearchCV(
        estimator, 
        param_distributions=param_dist, 
        n_iter=5, # Low iterations for speed
        cv=3, 
        scoring='accuracy', 
        random_state=42,
        n_jobs=-1
    )
    clf.fit(X_train_final, y)
    
    print(f"   -> Best Params for {name}: {clf.best_params_}")
    print(f"   -> Best Score: {clf.best_score_:.4f}")
    
    # Save the best model for later use
    best_estimators[name] = clf.best_estimator_

# 10. Comparison of model performances

In [ ]:
print("\n--- Comparison of Model Performances ---")

# 1. Visualization
plt.figure(figsize=(12, 6))
sns.barplot(data=results_df.sort_values(by='Accuracy', ascending=False), 
            x='Accuracy', y='Model', palette='viridis')

plt.title('Model Performance Comparison (Before Tuning)', fontsize=16)
plt.xlabel('Accuracy Score', fontsize=12)
plt.ylabel('Model', fontsize=12)
plt.xlim(0, 1.0)

# Add text labels
for index, value in enumerate(results_df.sort_values(by='Accuracy', ascending=False)['Accuracy']):
    plt.text(value, index, f'{value:.4f}', va='center')

plt.tight_layout()
plt.show()

# 2. DataFrame Display
print("\nDetailed Performance Table:")
print(results_df.sort_values(by='Accuracy', ascending=False).to_string(index=False))

# 3. Generate Submission using the absolute best model found during tuning
# Find the best of the tuned models
best_model_name = max(best_estimators, key=lambda k: best_estimators[k].score(X_train_final, y))
final_model = best_estimators[best_model_name]

print(f"\nGenerating submission with best tuned model: {best_model_name}")
final_model.fit(X_train_final, y) # Retrain on full data
preds = final_model.predict(X_test_final)

submission = pd.DataFrame({'id': test['id'], 'sentiment': preds})
submission.to_csv('submission.csv', index=False)
print("File 'submission.csv' saved successfully.")

In [ ]:
submission = pd.read_csv('/kaggle/working/submission.csv')
submission.isnull().sum()